# **nn.Module로 XOR 리팩토링하기**

## **1. 데이터 준비**

**Q1. XOR을 학습시킬 데이터를 생성합니다.**

In [ ]:
import torch
x_tensor = torch.tensor([[0,0], [1,0], [0,1], [1,1]], dtype=torch.float32)
y_tensor = torch.tensor([0, 1, 1, 0], dtype=torch.float32).reshape(4,1)

## **2. 신경망 생성하기**

**Q1. MLP 클래스를 만든다.**

- __init__() : Linear Layer 2개와 sigmoid 레이어 1개를 생성한다.
  - fc1 : 2개의 input과 2개의 output을 가진 fully connected layer
  - fc2 : 2개의 input과 1개의 output을 가진 fully connected layer
  - sigmoid : sigmoid 활성화 함수 layer
- forward() :
  - fc1 layer -> sigmoid -> fc2 layer -> sigmoid

In [8]:
import torch.nn as nn

class MLP(nn.Module):

    def __init__(self):
        # 제일 먼저 해야 하는 것 : super.init()
        super().__init__() # requires_grad=True 같이 weight 직접 안만든다 -> 알아서 해줌

        # layer 1 (입력 -> 은닉층)
        self.linear1 = nn.Linear(2, 2)
        self.sigmoid1 = nn.Sigmoid()
        # self.relu = nn.ReLU()

        # layer 2 (은닉층 -> 출력층)
        self.linear2 = nn.Linear(2, 1)
        self.sigmoid2 = nn.Sigmoid()


    def forward(self, X):
        z1 = self.linear1(X)    # 우리의 입력 데이터가 linear layer -> y=mx+b의 가중합 구하기
        a1 = self.sigmoid1(z1)      # 그리고 활성화
        z2 = self.linear2(a1)   # 다음층으로 값들 전달
        a2 = self.sigmoid2(z2)   # 활성화

        return a2               # 결과 (pred) return




## **3. Loss와 Gradient Descent**

**Q2. Loss 인스턴스를 만든다.**

criterion이라는 이름으로 nn.BCELoss의 인스턴스를 생성한다.

In [9]:
criterion = nn.BCELoss()    # binary cross Entropy

# 회귀 : MSELoss()
# 다중 분류 : 

**Q3. Gradient Descent를 구현한다.**

- torch.no_grad() : 그래프에 기록되는것을 방지한다.
- model.parameters() : weight, bias등의 가중치를 반환한다.

In [ ]:
def gradient_descent(model, lr):
    with torch.no_grad():
        for p in model.parameters():
            # p = p - (p.grad * lr)     # 새 텐서를 만들고, 변수 p가 새 텐서를 가리키게 함
            p -= (p.grad * lr)          # p가 가리키는 기존 텐서의 값 자체를 수정함
                   
                # p : 학습할 가중치
                # p.grad : 기울기 (가중치를 얼마나 수정할지 판단하는데 사용)

## **4. 모델 학습**

**Q3. 모델을 학습시킨다.**
- learning rate : 1.0
- epoch : 1000

forward -> loss -> backward -> gradient descent 순으로 1개의 epoch를 완성한다.
- 한번의 epoch 이후 다음 epoch 시작 전, gradient 초기화를 해줘야 한다.

100번의 epoch마다 loss를 출력하게 한다.

model은 train모드로 사용해야 한다.

In [15]:
epochs = 1000
lr = 1.0

model = MLP()

for epoch in range(epochs):

    model.train()                      # 모델 학습

    y_pred = model(x_tensor)           # 1. 순전파 (학습 데이터)
    loss = criterion(y_pred, y_tensor) # 2. 손실 계산 (※ ML과 param 순서 반대)

    loss.backward()                    # 3. 역전파 (loss 기반)
    gradient_descent(model, lr)        # 4. gradient_descent: 가중치 업데이트

    model.zero_grad()                  # 이전 기울기(gradient) 초기화
                                       # 가중치(weight)는 학습하면서 계속 유지 및 갱신


    if (epoch + 1) % 100 == 0:         # 100번 마다 loss 출력
        print(f"epoch {epoch}: {loss:.4f}")

epoch 99: 0.6728
epoch 199: 0.5758
epoch 299: 0.4873
epoch 399: 0.2597
epoch 499: 0.0963
epoch 599: 0.0527
epoch 699: 0.0354
epoch 799: 0.0264
epoch 899: 0.0210
epoch 999: 0.0174


## **5. 결과**

**Q1. 모델을 eval 모드로, 예측 결과를 확인해보자.**

In [17]:
import numpy as np

model.eval()        # 평가 모드

with torch.no_grad():
    y_pred = model(x_tensor)
    print(y_pred)
    print(np.where(y_pred>=0.6, 1, 0))
    print((y_pred >= 0.5).float())

tensor([[0.0136],
        [0.9813],
        [0.9813],
        [0.0177]])
[[0]
 [1]
 [1]
 [0]]
tensor([[0.],
        [1.],
        [1.],
        [0.]])
